# Laboratorio guiado: Apache Iceberg sobre HDFS e SeaweedFS

Este laboratorio desenvolve o percorrido do [tema 09](../09.Apache-iceberg.md): crear unha táboa, consultar datos, ampliar o esquema, engadir rexistros e recuperar un snapshot. A variante S3 mostra a evolución do particionamento.

## 1. Contorno e sesión

O clúster e JupyterLab deben estar iniciados segundo o [contorno de traballo](../02.construcion-escenarios.md#contorno-de-traballo-do-curso). A parte S3 precisa SeaweedFS e o bucket `spark`, descritos no [tema 07](../07.s3-seaweed.md).

**Antes de executar o laboratorio, o runtime Iceberg debe ser compatible con Spark 4.1.1.** O [apartado de configuración do tema 09](../09.Apache-iceberg.md#2-configuración-do-clúster) identifica o axuste pendente. Non se configura o catálogo nin se descargan dependencias desde o notebook.

Os bloques Python execútanse en orde nun notebook. Os datos créanse na sesión e non requiren ficheiros externos. Empregamos os catálogos existentes `local` (HDFS) e `minio` (S3 de SeaweedFS). A demostración usa un único escritor; os límites do catálogo Hadoop sobre S3A están explicados no tema 09.

Executa as celas Python en orde. Os comandos Bash e PowerShell dos bloques explicativos execútanse nos terminais indicados, fóra das celas Python.


In [ ]:
# 1. Contorno e sesión
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("apache-iceberg").getOrCreate()


## 2. Iceberg sobre HDFS

### 2.1. Namespace de exemplo

Un namespace novo en cada execución evita conflitos con táboas de demostración anteriores:

In [ ]:
# 2.1. Namespace de exemplo
from uuid import uuid4

namespace = f"meteo_{uuid4().hex[:8]}"
taboa_hdfs = f"local.{namespace}.lecturas"

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS local.{namespace}")
print("Táboa HDFS:", taboa_hdfs)


O catálogo Hadoop crea os directorios necesarios baixo o seu warehouse. O usuario da aplicación debe ter permiso para escribir nesa localización.

### 2.2. Datos iniciais e creación da táboa

O exemplo usa catro rexistros, cos tipos definidos. Para simplificar, a columna `data` contén datas como texto.

In [ ]:
# 2.2. Datos iniciais e creación da táboa
datos = [
    ("2024-01-01", "A Coruña", 11.2),
    ("2024-01-01", "Vigo", 13.5),
    ("2024-01-02", "A Coruña", 9.8),
    ("2024-01-02", "Vigo", 14.1),
]

df = spark.createDataFrame(
    datos,
    "data STRING, cidade STRING, temperatura DOUBLE",
)

df.writeTo(taboa_hdfs).using("iceberg").create()
spark.table(taboa_hdfs).show()


`create()` crea unha táboa nova e falla se xa existe. O nome xerado no apartado anterior permite repetir a demostración sen substituír outra táboa.

O snapshot inicial pode gardarse para consultalo máis adiante:

In [ ]:
# 2.2. Datos iniciais e creación da táboa
snapshot_inicial = spark.sql(
    f"SELECT snapshot_id FROM {taboa_hdfs}.snapshots ORDER BY committed_at"
).first()["snapshot_id"]

print("Snapshot inicial:", snapshot_inicial)


### 2.3. Consultas SQL

In [ ]:
# 2.3. Consultas SQL
spark.sql(f"""
    SELECT cidade, AVG(temperatura) AS media
    FROM {taboa_hdfs}
    GROUP BY cidade
    ORDER BY cidade
""").show()


A consulta devolve a temperatura media por cidade: **10.5** para A Coruña e **13.8** para Vigo.

### 2.4. Evolución do esquema

A incorporación dunha columna realízase con:

In [ ]:
# 2.4. Evolución do esquema
spark.sql(f"ALTER TABLE {taboa_hdfs} ADD COLUMN humidade DOUBLE")
spark.table(taboa_hdfs).printSchema()
spark.table(taboa_hdfs).show()


As catro filas anteriores teñen `NULL` en `humidade`. O cambio de esquema non necesita reescribir eses ficheiros nin engade por si só un snapshot de datos.

### 2.5. Inserción dun rexistro

In [ ]:
# 2.5. Inserción dun rexistro
spark.sql(f"""
    INSERT INTO {taboa_hdfs} (data, cidade, temperatura, humidade)
    VALUES ('2024-01-03', 'Lugo', 7.3, 65.0)
""")

spark.table(taboa_hdfs).orderBy("data", "cidade").show()


A táboa ten agora **cinco filas**. A lista de columnas fai explícito onde se insire cada valor.

### 2.6. Snapshots e time travel

As táboas de metadatos permiten consultar o histórico e os snapshots dispoñibles:

In [ ]:
# 2.6. Snapshots e time travel
spark.sql(f"SELECT * FROM {taboa_hdfs}.history").show(truncate=False)
spark.sql(f"SELECT * FROM {taboa_hdfs}.snapshots").show(truncate=False)


Para ler o estado inicial, úsase o identificador gardado tras a creación:

In [ ]:
# 2.6. Snapshots e time travel
spark.sql(f"""
    SELECT * FROM {taboa_hdfs}
    VERSION AS OF {snapshot_inicial}
""").show()


Esta lectura recupera os **catro rexistros iniciais**, sen o rexistro de Lugo. Non modifica a táboa actual.

En Iceberg, `VERSION AS OF` emprega un **identificador de snapshot**, non unha versión secuencial como en Delta. Tamén existe `TIMESTAMP AS OF` para consultar o estado nun momento determinado. O snapshot e os ficheiros necesarios deben seguir dispoñibles. [Consultas históricas e metadatos](https://iceberg.apache.org/docs/1.10.1/spark-queries/).

## 3. Iceberg sobre SeaweedFS

### 3.1. Creación da táboa

O escenario S3 usa o catálogo xa configurado **`minio`** e o bucket **`spark`**. SeaweedFS debe estar iniciado co clúster, como se explica no tema 07.

In [ ]:
# 3.1. Creación da táboa
taboa_s3 = f"minio.{namespace}.lecturas"

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS minio.{namespace}")
df.writeTo(taboa_s3).using("iceberg").create()

spark.table(taboa_s3).show()
print("Táboa S3:", taboa_s3)


Créase outra táboa cos catro rexistros do DataFrame inicial. Os datos e os metadatos quedan baixo `spark/iceberg-warehouse` en SeaweedFS.

### 3.2. Evolución do particionamento

O particionamento por cidade pode engadirse con:

In [ ]:
# 3.2. Evolución do particionamento
spark.sql(f"ALTER TABLE {taboa_s3} ADD PARTITION FIELD cidade")


O cambio afecta ás **novas escrituras**. Os ficheiros existentes manteñen a súa organización anterior; non se reescriben automaticamente.

Unha nova carga usa o novo particionamento:

In [ ]:
# 3.2. Evolución do particionamento
df_s3_novos = spark.createDataFrame(
    [("2024-01-03", "Lugo", 7.3), ("2024-01-03", "Vigo", 12.0)],
    "data STRING, cidade STRING, temperatura DOUBLE",
)

df_s3_novos.writeTo(taboa_s3).append()
spark.table(taboa_s3).orderBy("data", "cidade").show()


O resultado ten **seis filas**. As consultas seguen usando as columnas da táboa, sen indicar as rutas das particións. Iceberg tamén admite transformacións de particionamento, como agrupar unha columna de data por día. [Evolución do esquema e do particionamento](https://iceberg.apache.org/docs/1.10.1/spark-ddl/).

### 3.3. Consulta dos ficheiros

In [ ]:
# 3.3. Consulta dos ficheiros
spark.sql(f"""
    SELECT file_path, file_format, record_count, spec_id
    FROM {taboa_s3}.files
""").show(truncate=False)


`files` mostra os ficheiros do estado actual. `spec_id` identifica a especificación de particionamento, polo que poden coexistir ficheiros escritos antes e despois do cambio.

A Admin UI de SeaweedFS, en [http://localhost:9001](http://localhost:9001), permite explorar **Buckets → spark → iceberg-warehouse**. Gardar unha táboa aquí non a rexistra automaticamente na sección **Table Buckets**, que corresponde a outra integración de catálogo.

## 4. Unha carga diaria sobre HDFS

Despois de engadir `humidade`, as novas cargas inclúen esa columna co tipo definido:

In [ ]:
# 4. Unha carga diaria sobre HDFS
df_hoxe = spark.createDataFrame(
    [
        ("2024-01-04", "A Coruña", 10.4, 72.0),
        ("2024-01-04", "Vigo", 12.8, 68.0),
    ],
    "data STRING, cidade STRING, temperatura DOUBLE, humidade DOUBLE",
)

df_hoxe.writeTo(taboa_hdfs).append()
spark.table(taboa_hdfs).orderBy("data", "cidade").show()
spark.sql(f"SELECT * FROM {taboa_hdfs}.snapshots").show(truncate=False)


A táboa pasa a ter **sete filas**. `append()` engade rexistros: non actualiza os existentes nin elimina duplicados. Unha corrección dun rexistro anterior require unha operación como `UPDATE` ou `MERGE`.


## 5. Comprobación final

Tras completar o percorrido, a táboa HDFS ten sete filas e a táboa S3 seis. A consulta do snapshot inicial de HDFS conserva as catro observacións orixinais.

In [ ]:
# 5. Comprobación final
assert spark.table(taboa_hdfs).count() == 7
assert spark.table(taboa_s3).count() == 6
assert spark.sql(
    f"SELECT * FROM {taboa_hdfs} VERSION AS OF {snapshot_inicial}"
).count() == 4


Unha repetición completa empeza creando outro namespace. Repetir só unha inserción ou `append()` pode engadir duplicados e cambiar os recontos. Os resultados son os esperados; a execución completa no clúster segue pendente do axuste de compatibilidade e da validación.

A aplicación dos conceptos avalíase no [boletín Delta e Iceberg](../tarefas/tarefa-04-delta-iceberg.ipynb).